PARTE # 2 - ENTRENAMIENTO DEL MODELO YOLOV8 CON EL CONJUNTO DE DATOS DE LAS CAMARAS REALES

In [ ]:
#IMPORTACION DE LIBRERIAS

from pathlib import Path
import cv2
import pandas as pd



PREPARACION DEL CONJUNTO DE DATOS

In [ ]:

# =====================================================
# 1. RUTA DEL DATASET
# =====================================================
ROOT = Path("/mnt/d/PROYECTOS/INCENDIOS/DATA")

if not ROOT.exists():
    raise FileNotFoundError(
        f"No encuentro la carpeta: {ROOT}\n"
        "Comprueba que la unidad D: esté montada en WSL."
    )

EXT_VIDEO = {".mp4", ".avi", ".mkv", ".mov", ".m4v", ".ts"}
EXT_IMAGEN = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

videos = sorted(
    p for p in ROOT.rglob("*")
    if p.is_file() and p.suffix.lower() in EXT_VIDEO
)

imagenes = sorted(
    p for p in ROOT.rglob("*")
    if p.is_file() and p.suffix.lower() in EXT_IMAGEN
)

print("Carpeta raíz:", ROOT)
print("Total de vídeos:", len(videos))
print("Total de imágenes:", len(imagenes))

# =====================================================
# 2. ANALIZAR METADATOS DE LOS VÍDEOS
# =====================================================
registros = []

for i, video in enumerate(videos, start=1):
    cap = cv2.VideoCapture(str(video))

    if not cap.isOpened():
        registros.append({
            "archivo": video.name,
            "carpeta": video.parent.name,
            "duracion_min": None,
            "fps": None,
            "fotogramas": None,
            "resolucion": None,
            "estado": "No se pudo abrir"
        })
        cap.release()
        continue

    fps = cap.get(cv2.CAP_PROP_FPS)
    frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    ancho = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    alto = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # Evitar división por cero si el archivo no informa FPS válidos
    duracion_min = (
        frames / fps / 60
        if fps and fps > 0 and frames > 0
        else None
    )

    registros.append({
        "archivo": video.name,
        "carpeta": video.parent.name,
        "duracion_min": duracion_min,
        "fps": fps if fps > 0 else None,
        "fotogramas": frames,
        "resolucion": f"{ancho}x{alto}",
        "estado": "OK"
    })

    cap.release()

    if i % 10 == 0:
        print(f"Vídeos analizados: {i}/{len(videos)}")

df_videos = pd.DataFrame(registros)

# =====================================================
# 3. MOSTRAR RESULTADOS
# =====================================================
if not df_videos.empty:
    print("\n========== INVENTARIO DE VÍDEOS ==========")
    print(df_videos.round(2).to_string(index=False))

    print("\n========== RESUMEN POR CARPETA ==========")

    resumen = df_videos.groupby("carpeta").agg(
        cantidad_videos=("archivo", "count"),
        duracion_total_min=("duracion_min", "sum"),
        resoluciones=("resolucion", "nunique"),
        errores=("estado", lambda s: (s != "OK").sum())
    )

    print(resumen.round(2).to_string())

# =====================================================
# 4. REVISAR LAS 7 IMÁGENES POSITIVAS Y SUS RUTAS
# =====================================================
carpeta_positivos = ROOT / "positivos"

if carpeta_positivos.exists():
    imgs_positivas = [
        p for p in carpeta_positivos.rglob("*")
        if p.is_file() and p.suffix.lower() in EXT_IMAGEN
    ]

    vids_positivos = [
        p for p in carpeta_positivos.rglob("*")
        if p.is_file() and p.suffix.lower() in EXT_VIDEO
    ]

    print("\n========== CARPETA POSITIVOS ==========")
    print("Vídeos:", len(vids_positivos))
    print("Imágenes:", len(imgs_positivas))

    print("\nNombres de las imágenes:")
    for imagen in imgs_positivas:
        print(" -", imagen.name)
else:
    print("\nNo se encontró una carpeta llamada 'positivos'.")
    print("Revisa la capitalización o el nombre de la carpeta.")

EXTRACCION DE FOTOGRAMAS DE LOS VIDEOS


In [ ]:
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import math
import shutil

# =====================================================
# 1. RUTAS
# =====================================================
ROOT = Path("/mnt/d/PROYECTOS/INCENDIOS/DATA")

# Guardar los resultados FUERA de la carpeta original
OUTPUT = ROOT.parent / "DATA_PROCESADA_SYLVA"

CARPETA_NEGATIVOS = OUTPUT / "candidatos" / "negativos"
CARPETA_POSITIVOS_VIDEO = (
    OUTPUT / "candidatos" / "positivos_video_revision"
)
CARPETA_POSITIVOS_IMAGENES = (
    OUTPUT / "candidatos" / "positivos_imagenes"
)

assert ROOT.exists(), f"No existe la ruta: {ROOT}"

# =====================================================
# 2. CONFIGURACIÓN
# =====================================================
INTERVALO_NEGATIVOS = 30.0   # segundos
INTERVALO_POSITIVOS = 2.0    # segundos

# Límite para no generar cantidades excesivas por vídeo.
MAX_NEGATIVOS_POR_VIDEO = 400
MAX_POSITIVOS_POR_VIDEO = 600

# Primera ejecución: solo estimar.
# Después de revisar las cantidades, cambiar a False.
DRY_RUN = False

EXT_VIDEO = {".mp4", ".avi", ".mkv", ".mov", ".m4v", ".ts"}
EXT_IMAGEN = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# =====================================================
# 3. ENCONTRAR Y CLASIFICAR VÍDEOS
# =====================================================
videos = sorted(
    p for p in ROOT.rglob("*")
    if p.is_file() and p.suffix.lower() in EXT_VIDEO
)

def es_positivo(video):
    return any(
        parte.name.casefold() == "positivos"
        for parte in video.parents
    )

planes = []

for video in videos:
    positivo = es_positivo(video)

    cap = cv2.VideoCapture(str(video))

    if not cap.isOpened():
        print("AVISO: no se pudo abrir:", video)
        cap.release()
        continue

    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()

    if fps <= 0 or total_frames <= 0:
        print("AVISO: metadatos no válidos:", video)
        continue

    duracion = total_frames / fps

    intervalo_base = (
        INTERVALO_POSITIVOS if positivo
        else INTERVALO_NEGATIVOS
    )

    max_frames = (
        MAX_POSITIVOS_POR_VIDEO if positivo
        else MAX_NEGATIVOS_POR_VIDEO
    )

    # Adaptar el intervalo en vídeos especialmente largos.
    intervalo = max(intervalo_base, duracion / max_frames)

    cantidad_estimada = math.ceil(duracion / intervalo)

    # Crear un nombre basado en la ruta relativa del vídeo.
    prefijo = "_".join(
        video.relative_to(ROOT).with_suffix("").parts
    ).replace(" ", "_")

    planes.append({
        "video": video,
        "prefijo": prefijo,
        "grupo": "positivo_video_revision" if positivo else "negativo",
        "duracion_min": duracion / 60,
        "fps": fps,
        "resolucion": None,
        "intervalo_seg": intervalo,
        "fotogramas_estimados": cantidad_estimada,
    })

df_plan = pd.DataFrame([
    {
        "video": p["video"].name,
        "grupo": p["grupo"],
        "duracion_min": p["duracion_min"],
        "intervalo_seg": p["intervalo_seg"],
        "fotogramas_estimados": p["fotogramas_estimados"],
    }
    for p in planes
])

print("========== PLAN DE EXTRACCIÓN ==========")
print("Vídeos encontrados:", len(videos))
print("Vídeos procesables:", len(planes))

if not df_plan.empty:
    print("\nResumen por grupo:")

    print(
        df_plan.groupby("grupo").agg(
            videos=("video", "count"),
            duracion_total_min=("duracion_min", "sum"),
            imagenes_estimadas=("fotogramas_estimados", "sum")
        ).round(1).to_string()
    )

# =====================================================
# 4. EXTRAER SI SE DESACTIVA DRY_RUN
# =====================================================
if DRY_RUN:
    print("\nMODO DE PRUEBA: no se han extraído imágenes.")
    print("Revisa los conteos anteriores.")
    print("Cuando estés conforme, cambia DRY_RUN = False.")
else:
    for carpeta in (
        CARPETA_NEGATIVOS,
        CARPETA_POSITIVOS_VIDEO,
        CARPETA_POSITIVOS_IMAGENES
    ):
        carpeta.mkdir(parents=True, exist_ok=True)

    manifest = []

    for p in planes:
        video = p["video"]
        grupo = p["grupo"]

        destino = (
            CARPETA_NEGATIVOS
            if grupo == "negativo"
            else CARPETA_POSITIVOS_VIDEO
        )

        cap = cv2.VideoCapture(str(video))

        if not cap.isOpened():
            print("No se pudo abrir durante extracción:", video)
            continue

        duracion = p["duracion_min"] * 60
        intervalo = p["intervalo_seg"]

        for t in np.arange(0, duracion, intervalo):
            cap.set(cv2.CAP_PROP_POS_MSEC, float(t) * 1000)
            ok, frame = cap.read()

            if not ok or frame is None:
                continue

            archivo = (
                f"{p['prefijo']}_t{int(t):07d}s.jpg"
            )
            salida = destino / archivo

            guardado = cv2.imwrite(
                str(salida),
                frame,
                [cv2.IMWRITE_JPEG_QUALITY, 95]
            )

            if guardado:
                manifest.append({
                    "archivo": str(salida),
                    "grupo_candidato": grupo,
                    "video_origen": str(video),
                    "segundo_aproximado": round(float(t), 2),
                    "requiere_revision": grupo.startswith("positivo"),
                })

        cap.release()

    # Copiar las imágenes existentes de la carpeta positivos.
    carpeta_origen_positivos = next(
        (
            p for p in ROOT.iterdir()
            if p.is_dir() and p.name.casefold() == "positivos"
        ),
        None
    )

    if carpeta_origen_positivos is not None:
        imagenes_existentes = [
            p for p in carpeta_origen_positivos.rglob("*")
            if p.is_file() and p.suffix.lower() in EXT_IMAGEN
        ]

        for i, img in enumerate(imagenes_existentes, start=1):
            nombre = f"POSIMG_{i:03d}_{img.name}"
            salida = CARPETA_POSITIVOS_IMAGENES / nombre

            shutil.copy2(img, salida)

            manifest.append({
                "archivo": str(salida),
                "grupo_candidato": "positiva_imagen_original",
                "video_origen": str(img),
                "segundo_aproximado": None,
                "requiere_revision": True,
            })

    ruta_manifest = OUTPUT / "manifest_frames.csv"
    OUTPUT.mkdir(parents=True, exist_ok=True)

    pd.DataFrame(manifest).to_csv(
        ruta_manifest,
        index=False,
        encoding="utf-8"
    )

    print("\n========== EXTRACCIÓN TERMINADA ==========")
    print("Fotogramas y copias registrados:", len(manifest))
    print("Manifiesto:", ruta_manifest)
    print("Carpeta de salida:", OUTPUT)

In [ ]:
from pathlib import Path
import pandas as pd

# Carpeta donde guardamos los fotogramas extraídos
OUTPUT = Path("/mnt/d/PROYECTOS/INCENDIOS/DATA_PROCESADA_SYLVA")

carpetas = {
    "Negativos": OUTPUT / "candidatos" / "negativos",
    "Positivos de video": (
        OUTPUT / "candidatos" / "positivos_video_revision"
    ),
    "Imagenes positivas originales": (
        OUTPUT / "candidatos" / "positivos_imagenes"
    ),
}

EXTENSIONES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

print("========== INVENTARIO DE FOTOGRAMAS ==========")

for nombre, carpeta in carpetas.items():
    archivos = [
        p for p in carpeta.rglob("*")
        if p.is_file() and p.suffix.lower() in EXTENSIONES
    ] if carpeta.exists() else []

    print(f"{nombre}: {len(archivos):,} imágenes")

# Consultar el manifiesto de extracción, si existe
manifest_path = OUTPUT / "manifest_frames.csv"

if manifest_path.exists():
    manifest = pd.read_csv(manifest_path)

    print("\n========== PROCEDENCIA DE LOS FOTOGRAMAS ==========")
    print(
        manifest.groupby("grupo_candidato")
        .agg(
            cantidad=("archivo", "count"),
            videos_origen=("video_origen", "nunique")
        )
        .to_string()
    )

    print("\nManifiesto:", manifest_path)
else:
    print("\nNo se encontró manifest_frames.csv.")

In [ ]:
#REVISION DE IMAGENES 
import random
import math
import cv2
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

random.seed(42)

def mostrar_mosaico(rutas, titulo, max_imagenes=24, columnas=4):
    rutas = [Path(p) for p in rutas if Path(p).exists()]

    if not rutas:
        print(f"No hay imágenes para mostrar: {titulo}")
        return

    if len(rutas) > max_imagenes:
        rutas = random.sample(rutas, max_imagenes)

    filas = math.ceil(len(rutas) / columnas)

    fig, axes = plt.subplots(
        filas, columnas,
        figsize=(16, 4 * filas),
        squeeze=False
    )

    for ax in axes.flat:
        ax.axis("off")

    for ax, ruta in zip(axes.flat, rutas):
        imagen = cv2.imread(str(ruta))

        if imagen is None:
            ax.set_title("No se pudo leer")
            continue

        imagen = cv2.cvtColor(imagen, cv2.COLOR_BGR2RGB)
        ax.imshow(imagen)
        ax.set_title(ruta.name[:36], fontsize=8)
        ax.axis("off")

    fig.suptitle(titulo, fontsize=15)
    plt.tight_layout()
    plt.show()


# 1. Muestra de escenas negativas
carpeta_neg = OUTPUT / "candidatos" / "negativos"
imagenes_neg = list(carpeta_neg.glob("*.jpg"))

mostrar_mosaico(
    imagenes_neg,
    "Negativos candidatos: niebla, día, noche y otras escenas",
    max_imagenes=24
)

# 2. Fotogramas de los vídeos positivos
carpeta_pos_video = (
    OUTPUT / "candidatos" / "positivos_video_revision"
)
imagenes_pos_video = list(carpeta_pos_video.glob("*.jpg"))

mostrar_mosaico(
    imagenes_pos_video,
    "Fotogramas de los vídeos positivos: REVISAR",
    max_imagenes=24
)

# 3. Imágenes positivas originales
carpeta_pos_original = (
    OUTPUT / "candidatos" / "positivos_imagenes"
)
imagenes_pos_originales = list(carpeta_pos_original.glob("*"))

mostrar_mosaico(
    imagenes_pos_originales,
    "Imágenes positivas originales",
    max_imagenes=20
)

CLASIFICACION DE FOTOGRAMAS EXTRAIDOS DE LOS VIDEOS POSITIVOS

Despues de observar los fotogramas extraidos de los videos Positivos se encontraron imagenes sin presencia de fuego o de humo ya que al ser extraidos de un video este al momento de reproducirse la presencia de humo o fuego va apareciendo por lo que hay escenas que no tiene presencia de humo o de fuego.
Asi que se deben de clasificar estos fotogramas 

In [ ]:

OUTPUT = Path(
    "/mnt/d/PROYECTOS/INCENDIOS/DATA_PROCESADA_SYLVA"
)

ruta_manifest = OUTPUT / "manifest_frames.csv"
ruta_revision = OUTPUT / "revision_fotogramas_positivos.csv"

if not ruta_manifest.exists():
    raise FileNotFoundError(
        f"No se encontró el manifiesto: {ruta_manifest}"
    )

manifest = pd.read_csv(ruta_manifest)

# Seleccionar únicamente los fotogramas de vídeos positivos
revision = manifest[
    manifest["grupo_candidato"] == "positivo_video_revision"
].copy()

if revision.empty:
    raise ValueError(
        "No se encontraron fotogramas del grupo positivo_video_revision."
    )

# Columnas para documentar la revisión humana
revision["decision_revision"] = ""
revision["observaciones"] = ""

# Orden útil para revisar la secuencia de cada vídeo
revision = revision.sort_values(
    ["video_origen", "segundo_aproximado"]
)

revision.to_csv(
    ruta_revision,
    index=False,
    encoding="utf-8-sig"
)

print("========== ARCHIVO DE REVISIÓN ==========")
print("Fotogramas por revisar:", len(revision))
print("Vídeos de origen:", revision["video_origen"].nunique())
print("CSV creado:", ruta_revision)

print("\nCantidad de fotogramas por vídeo:")
print(
    revision.groupby("video_origen")
    .size()
    .rename("fotogramas")
    .to_string()
)

print("\nValores permitidos en decision_revision:")
print("background, smoke, fire, fire_smoke, uncertain")

In [ ]:
# ============================================================
# GALERÍA DE REVISIÓN DE FOTOGRAMAS POSITIVOS
# Sylva AI - Etapa 2: datos de cámaras DVR
# ============================================================

from pathlib import Path
from IPython.display import display
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import math


# ============================================================
# 1. CONFIGURAR RUTAS
# ============================================================

OUTPUT = Path(
    "/mnt/d/PROYECTOS/INCENDIOS/DATA_PROCESADA_SYLVA"
)

# CSV original: solamente lo leeremos
ruta_original = OUTPUT / "revision_fotogramas_positivos.csv"

# Guardar el nuevo CSV fuera de la unidad D:
CARPETA_REVISION = (
    Path.home() / "sylva_revision"
)

CARPETA_REVISION.mkdir(
    parents=True,
    exist_ok=True
)

ruta_revision = (
    CARPETA_REVISION
    / "revision_fotogramas_positivos_con_ids.csv"
)


# ============================================================
# 2. COMPROBAR Y LEER EL CSV
# ============================================================

if not ruta_original.exists():
    raise FileNotFoundError(
        f"No se encontró el archivo original:\n{ruta_original}"
    )

df = pd.read_csv(ruta_original)

columnas_necesarias = {
    "archivo",
    "video_origen",
    "segundo_aproximado"
}

faltantes = columnas_necesarias - set(df.columns)

if faltantes:
    raise ValueError(
        f"Faltan columnas necesarias en el CSV: {faltantes}"
    )

# Asegurar que los segundos sean numéricos
df["segundo_aproximado"] = pd.to_numeric(
    df["segundo_aproximado"],
    errors="coerce"
)

# Ordenar por vídeo y tiempo de extracción
df = df.sort_values(
    ["video_origen", "segundo_aproximado"]
).reset_index(drop=True)


# ============================================================
# 3. PREPARAR LAS COLUMNAS DE REVISIÓN
# ============================================================

# Conservar las decisiones existentes si ya se habían registrado
if "decision_revision" not in df.columns:
    df["decision_revision"] = ""

if "observaciones" not in df.columns:
    df["observaciones"] = ""

# Asignar un identificador a cada fila
df["id_revision"] = range(1, len(df) + 1)

# Guardar primero las columnas principales para facilitar su uso
columnas_inicio = [
    "id_revision",
    "archivo",
    "video_origen",
    "segundo_aproximado",
    "decision_revision",
    "observaciones"
]

otras_columnas = [
    columna
    for columna in df.columns
    if columna not in columnas_inicio
]

df = df[columnas_inicio + otras_columnas]


# ============================================================
# 4. GUARDAR UNA COPIA DEL CSV
# ============================================================

df.to_csv(
    ruta_revision,
    index=False,
    encoding="utf-8-sig"
)

print("=" * 65)
print("CSV DE REVISIÓN PREPARADO")
print("=" * 65)

print(f"Total de fotogramas: {len(df)}")
print(f"Vídeos de origen: {df['video_origen'].nunique()}")
print(f"CSV original conservado:\n{ruta_original}")
print(f"\nNueva copia guardada en:\n{ruta_revision}")


# ============================================================
# 5. FUNCIÓN PARA MOSTRAR UNA GALERÍA
# ============================================================

def mostrar_galerias(
    df_video,
    tamano_grupo=16,
    columnas=4
):
    """
    Muestra los fotogramas de un vídeo en mosaicos.
    Cada imagen incluye su ID y el segundo de extracción.
    """

    registros = df_video.to_dict("records")

    if not registros:
        print("No hay fotogramas para mostrar.")
        return

    total_grupos = math.ceil(
        len(registros) / tamano_grupo
    )

    for inicio in range(
        0,
        len(registros),
        tamano_grupo
    ):

        grupo = registros[
            inicio:inicio + tamano_grupo
        ]

        filas = math.ceil(
            len(grupo) / columnas
        )

        fig, axes = plt.subplots(
            filas,
            columnas,
            figsize=(16, filas * 3.5),
            squeeze=False
        )

        # Ocultar todos los ejes inicialmente
        for ax in axes.flat:
            ax.axis("off")

        for ax, registro in zip(
            axes.flat,
            grupo
        ):

            ruta_img = Path(
                str(registro["archivo"])
            )

            id_img = registro["id_revision"]
            segundo = registro["segundo_aproximado"]

            # Comprobar que el fotograma exista
            if not ruta_img.exists():
                ax.set_title(
                    f"ID {id_img:03d}\nArchivo no encontrado",
                    fontsize=9
                )
                continue

            # Leer imagen
            imagen = cv2.imread(str(ruta_img))

            if imagen is None:
                ax.set_title(
                    f"ID {id_img:03d}\nNo se pudo leer",
                    fontsize=9
                )
                continue

            # OpenCV usa BGR; Matplotlib necesita RGB
            imagen = cv2.cvtColor(
                imagen,
                cv2.COLOR_BGR2RGB
            )

            ax.imshow(imagen)

            ax.set_title(
                f"ID {id_img:03d} | t = {segundo:.1f} s",
                fontsize=10
            )

            ax.axis("off")

        nombre_video = Path(
            str(grupo[0]["video_origen"])
        ).name

        numero_grupo = (
            inicio // tamano_grupo
        ) + 1

        fig.suptitle(
            f"{nombre_video} | "
            f"Grupo {numero_grupo}/{total_grupos}",
            fontsize=15
        )

        plt.tight_layout()
        display(fig)
        plt.close(fig)


# ============================================================
# 6. MOSTRAR TODOS LOS VÍDEOS POSITIVOS
# ============================================================

print("\n")
print("=" * 65)
print("GALERÍA DE FOTOGRAMAS POR VÍDEO")
print("=" * 65)

for video, grupo in df.groupby(
    "video_origen",
    sort=True
):

    print("\n" + "-" * 65)
    print(f"VÍDEO: {Path(str(video)).name}")
    print(f"Fotogramas que revisar: {len(grupo)}")
    print("-" * 65)

    mostrar_galerias(
        grupo,
        tamano_grupo=16,
        columnas=4
    )


# ============================================================
# 7. RESUMEN DE REVISIÓN
# ============================================================

print("\n")
print("=" * 65)
print("RESUMEN FINAL")
print("=" * 65)

print("Total de fotogramas:", len(df))
print("Total de vídeos:", df["video_origen"].nunique())

print("\nDecisiones registradas actualmente:")

print(
    df["decision_revision"]
    .fillna("(sin revisar)")
    .replace("", "(sin revisar)")
    .value_counts()
    .to_string()
)

print("\nCSV para registrar tus decisiones:")
print(ruta_revision)

CLASIFICADOR INTERACTIVO DE FOTOGRAMAS

In [ ]:
from pathlib import Path
import pandas as pd
import ipywidgets as widgets

from IPython.display import (
    display,
    clear_output,
    Image as IPyImage
)

# =====================================================
# 1. RUTAS
# =====================================================

CARPETA_REVISION = Path.home() / "sylva_revision"

RUTA_CSV = (
    CARPETA_REVISION
    / "revision_fotogramas_positivos_con_ids.csv"
)

if not RUTA_CSV.exists():
    raise FileNotFoundError(
        f"No se encontró el CSV: {RUTA_CSV}"
    )

df_revision = pd.read_csv(RUTA_CSV)

# Crear las columnas si no existen
for columna in ["decision_revision", "observaciones"]:
    if columna not in df_revision.columns:
        df_revision[columna] = ""

df_revision["decision_revision"] = (
    df_revision["decision_revision"].fillna("")
)

df_revision["observaciones"] = (
    df_revision["observaciones"].fillna("")
)

# =====================================================
# 2. CATEGORÍAS DISPONIBLES
# =====================================================

OPCIONES = [
    ("-- Selecciona una categoría --", ""),
    ("Background: sin humo ni fuego", "background"),
    ("Smoke: humo visible", "smoke"),
    ("Fire: fuego visible", "fire"),
    ("Fire + smoke: ambos visibles", "fire_smoke"),
    ("Uncertain: no estoy segura", "uncertain"),
]

# Índice de la imagen actual
estado = {"indice": 0}

# =====================================================
# 3. CONTROLES DE LA INTERFAZ
# =====================================================

titulo = widgets.HTML()

imagen_salida = widgets.Output()

decision = widgets.Dropdown(
    options=OPCIONES,
    value="",
    description="Categoría:",
    layout=widgets.Layout(width="500px")
)

observaciones = widgets.Textarea(
    value="",
    placeholder="Observaciones opcionales...",
    description="Notas:",
    layout=widgets.Layout(width="700px", height="75px")
)

boton_anterior = widgets.Button(
    description="← Anterior",
    button_style="",
    icon="arrow-left"
)

boton_guardar = widgets.Button(
    description="Guardar",
    button_style="success",
    icon="save"
)

boton_siguiente = widgets.Button(
    description="Guardar y siguiente →",
    button_style="primary",
    icon="arrow-right"
)

estado_texto = widgets.HTML()


# =====================================================
# 4. GUARDAR LA DECISIÓN ACTUAL
# =====================================================

def guardar_decision():
    i = estado["indice"]

    df_revision.at[i, "decision_revision"] = decision.value
    df_revision.at[i, "observaciones"] = observaciones.value

    df_revision.to_csv(
        RUTA_CSV,
        index=False,
        encoding="utf-8-sig"
    )

    estado_texto.value = (
        "<b>Guardado correctamente.</b> "
        f"Archivo: {RUTA_CSV.name}"
    )


# =====================================================
# 5. MOSTRAR LA IMAGEN ACTUAL
# =====================================================

def mostrar_actual():
    i = estado["indice"]
    fila = df_revision.iloc[i]

    ruta_img = Path(str(fila["archivo"]))

    titulo.value = (
        f"<h3>Fotograma {i + 1} de {len(df_revision)}</h3>"
        f"<b>ID:</b> {fila.get('id_revision', i + 1)}"
        f"<br><b>Vídeo:</b> {Path(str(fila['video_origen'])).name}"
        f"<br><b>Segundo aproximado:</b> "
        f"{fila['segundo_aproximado']} s"
        f"<br><b>Archivo:</b> {ruta_img.name}"
    )

    # Recuperar una decisión guardada anteriormente
    valor_guardado = str(fila["decision_revision"])

    valores_validos = [opcion[1] for opcion in OPCIONES]

    if valor_guardado not in valores_validos:
        valor_guardado = ""

    decision.value = valor_guardado
    observaciones.value = str(fila["observaciones"])

    with imagen_salida:
        clear_output(wait=True)

        if ruta_img.exists():
            display(
                IPyImage(
                    filename=str(ruta_img),
                    width=900
                )
            )
        else:
            print(f"No se encontró la imagen:\n{ruta_img}")

    estado_texto.value = ""


# =====================================================
# 6. NAVEGACIÓN
# =====================================================

def ir_anterior(_):
    if estado["indice"] > 0:
        guardar_decision()
        estado["indice"] -= 1
        mostrar_actual()


def guardar_actual(_):
    guardar_decision()


def ir_siguiente(_):
    guardar_decision()

    if estado["indice"] < len(df_revision) - 1:
        estado["indice"] += 1
        mostrar_actual()
    else:
        estado_texto.value = (
            "<b>¡Has llegado al último fotograma!</b> "
            "Tus decisiones están guardadas."
        )


boton_anterior.on_click(ir_anterior)
boton_guardar.on_click(guardar_actual)
boton_siguiente.on_click(ir_siguiente)


# =====================================================
# 7. MOSTRAR LA INTERFAZ
# =====================================================

display(
    widgets.VBox([
        titulo,
        imagen_salida,
        decision,
        observaciones,
        widgets.HBox([
            boton_anterior,
            boton_guardar,
            boton_siguiente
        ]),
        estado_texto
    ])
)

mostrar_actual()

print(f"CSV de revisión: {RUTA_CSV}")

RESUMEN DE CLASIFICACION

In [ ]:
from pathlib import Path
import pandas as pd

# CSV donde guardamos las decisiones
RUTA_CSV = (
    Path.home()
    / "sylva_revision"
    / "revision_fotogramas_positivos_con_ids.csv"
)

df = pd.read_csv(RUTA_CSV)

# Normalizar las decisiones para evitar espacios accidentales
df["decision_revision"] = (
    df["decision_revision"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

print("========== RESUMEN DE CLASIFICACIÓN ==========")
print("Total de fotogramas:", len(df))
print("Vídeos de origen:", df["video_origen"].nunique())

print("\nCantidad por categoría:")
print(
    df["decision_revision"]
    .replace("", "sin revisar")
    .value_counts()
    .to_string()
)

print("\n========== RESUMEN POR VÍDEO ==========")

tabla = pd.crosstab(
    df["video_origen"].apply(lambda x: Path(x).name),
    df["decision_revision"].replace("", "sin revisar")
)

print(tabla.to_string())

# Identificar registros pendientes
pendientes = df[
    df["decision_revision"].isin(["", "uncertain"])
]

print("\nFotogramas pendientes o inciertos:", len(pendientes))

if len(pendientes) > 0:
    print("\nPrimeros registros pendientes:")
    print(
        pendientes[
            ["id_revision", "archivo", "decision_revision"]
        ].head(10).to_string(index=False)
    )

PREPARACION DE LAS IMAGENES PARA ETIQUETARLAS

In [ ]:
from pathlib import Path
import pandas as pd
import shutil

# =====================================================
# 1. RUTAS
# =====================================================

OUTPUT = Path(
    "/mnt/d/PROYECTOS/INCENDIOS/DATA_PROCESADA_SYLVA"
)

CSV_REVISION = (
    Path.home()
    / "sylva_revision"
    / "revision_fotogramas_positivos_con_ids.csv"
)

CARPETA_POSITIVOS_ORIGINALES = (
    OUTPUT / "candidatos" / "positivos_imagenes"
)

DESTINO = OUTPUT / "etiquetado_dvr"

CARPETA_ANOTAR = DESTINO / "imagenes_para_anotar"
CARPETA_FONDOS = DESTINO / "fondos_verificados"

CARPETA_ANOTAR.mkdir(parents=True, exist_ok=True)
CARPETA_FONDOS.mkdir(parents=True, exist_ok=True)

assert CSV_REVISION.exists(), (
    f"No se encontró el CSV: {CSV_REVISION}"
)

df = pd.read_csv(CSV_REVISION)

# Normalizar las decisiones
df["decision_revision"] = (
    df["decision_revision"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

EXTENSIONES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# Categorías que contienen algún objeto que requiere cajas
CATEGORIAS_POSITIVAS = {
    "smoke",
    "fire",
    "fire_smoke"
}

manifest_salida = []
copiados_positivos = 0
copiados_fondos = 0
omitidos = []

# =====================================================
# 2. COPIAR FOTOGRAMAS REVISADOS
# =====================================================

for _, fila in df.iterrows():

    origen = Path(str(fila["archivo"]))
    decision = fila["decision_revision"]

    if not origen.exists():
        omitidos.append(str(origen))
        continue

    id_revision = int(fila["id_revision"])
    video = Path(str(fila["video_origen"])).stem

    segundo = pd.to_numeric(
        fila["segundo_aproximado"],
        errors="coerce"
    )

    segundo_str = (
        f"{int(segundo):07d}s"
        if pd.notna(segundo)
        else "tiempo_desconocido"
    )

    nombre_salida = (
        f"ID{id_revision:03d}_{video}_{segundo_str}"
        f"{origen.suffix.lower()}"
    )

    # Positivos: necesitamos dibujar cajas
    if decision in CATEGORIAS_POSITIVAS:

        destino = CARPETA_ANOTAR / nombre_salida
        shutil.copy2(origen, destino)
        copiados_positivos += 1

        manifest_salida.append({
            "archivo": nombre_salida,
            "ruta": str(destino),
            "tipo": "fotograma_positivo",
            "decision_previa": decision,
            "requiere_cajas": True,
            "video_origen": str(fila["video_origen"]),
            "segundo_aproximado": segundo
        })

    # Fondos: imágenes sin objetos por anotar
    elif decision == "background":

        destino = CARPETA_FONDOS / nombre_salida
        shutil.copy2(origen, destino)
        copiados_fondos += 1

        manifest_salida.append({
            "archivo": nombre_salida,
            "ruta": str(destino),
            "tipo": "fondo_verificado",
            "decision_previa": decision,
            "requiere_cajas": False,
            "video_origen": str(fila["video_origen"]),
            "segundo_aproximado": segundo
        })

    elif decision == "uncertain" or decision == "":
        omitidos.append(str(origen))

# =====================================================
# 3. INCORPORAR LAS 7 IMÁGENES ORIGINALES
# =====================================================

imagenes_originales = []

if CARPETA_POSITIVOS_ORIGINALES.exists():
    imagenes_originales = sorted(
        p for p in CARPETA_POSITIVOS_ORIGINALES.rglob("*")
        if p.is_file() and p.suffix.lower() in EXTENSIONES
    )

for i, origen in enumerate(imagenes_originales, start=1):

    nombre_salida = (
        f"IMG_ORIGINAL_{i:03d}_{origen.name}"
    )

    destino = CARPETA_ANOTAR / nombre_salida
    shutil.copy2(origen, destino)

    manifest_salida.append({
        "archivo": nombre_salida,
        "ruta": str(destino),
        "tipo": "imagen_positiva_original",
        "decision_previa": "revisar_clase",
        "requiere_cajas": True,
        "video_origen": "",
        "segundo_aproximado": None
    })

# =====================================================
# 4. GUARDAR EL MANIFIESTO
# =====================================================

ruta_manifest_salida = (
    DESTINO / "manifest_para_etiquetado.csv"
)

df_manifest = pd.DataFrame(manifest_salida)

df_manifest.to_csv(
    ruta_manifest_salida,
    index=False,
    encoding="utf-8-sig"
)

print("========== PREPARACIÓN COMPLETADA ==========")
print("Fotogramas positivos copiados:", copiados_positivos)
print("Fondos verificados copiados:", copiados_fondos)
print("Imágenes originales añadidas:", len(imagenes_originales))
print("Archivos omitidos o pendientes:", len(omitidos))

print("\nCarpeta para dibujar cajas:")
print(CARPETA_ANOTAR)

print("\nCarpeta de fondos:")
print(CARPETA_FONDOS)

print("\nManifiesto:")
print(ruta_manifest_salida)

print("\nContenido de la carpeta de anotación:")
print("Total:", len(list(CARPETA_ANOTAR.iterdir())), "archivos")

if omitidos:
    print("\nAVISO: algunos archivos no se copiaron.")
    print("Revisa las rutas de la lista de omitidos.")

PASAR EL CONJUNTO DE DATOS A GITHUB SIN CAJAS DELIMITADORAS

In [ ]:
from pathlib import Path
import pandas as pd
import shutil

# Fuentes locales
OUTPUT = Path(
    "/mnt/d/PROYECTOS/INCENDIOS/DATA_PROCESADA_SYLVA"
)

CARPETA_POSITIVOS = (
    OUTPUT / "etiquetado_dvr" / "imagenes_para_anotar"
)
CARPETA_FONDOS = (
    OUTPUT / "etiquetado_dvr" / "fondos_verificados"
)
MANIFEST_ORIGEN = (
    OUTPUT / "etiquetado_dvr" / "manifest_para_etiquetado.csv"
)

# Destino dentro del repositorio Git
REPO = Path.home() / "sylva-ai-fire-smoke"
DESTINO = REPO / "dataset_versions" / "dvr_stage2"
CARPETA_IMAGENES = DESTINO / "images"

# Comprobar fuentes
for ruta in [
    REPO,
    CARPETA_POSITIVOS,
    CARPETA_FONDOS,
    MANIFEST_ORIGEN
]:
    if not ruta.exists():
        raise FileNotFoundError(f"No existe: {ruta}")

# No sobrescribir accidentalmente una versión ya preparada
if CARPETA_IMAGENES.exists() and any(CARPETA_IMAGENES.iterdir()):
    raise FileExistsError(
        f"La carpeta ya contiene archivos: {CARPETA_IMAGENES}\n"
        "No se modificó para proteger los datos existentes."
    )

CARPETA_IMAGENES.mkdir(parents=True, exist_ok=True)

extensiones = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
archivos_copiados = []

# Copiar imágenes positivas candidatas y originales
for carpeta in [CARPETA_POSITIVOS, CARPETA_FONDOS]:
    for origen in sorted(carpeta.iterdir()):
        if origen.is_file() and origen.suffix.lower() in extensiones:
            destino = CARPETA_IMAGENES / origen.name

            if destino.exists():
                raise FileExistsError(
                    f"Nombre de archivo duplicado: {destino}"
                )

            shutil.copy2(origen, destino)
            archivos_copiados.append(origen.name)

# Crear un manifiesto limpio, sin rutas absolutas del equipo
manifest = pd.read_csv(MANIFEST_ORIGEN)

def nombre_video(valor):
    if pd.isna(valor) or not str(valor).strip():
        return ""
    return Path(str(valor)).name

manifest_limpio = pd.DataFrame({
    "filename": manifest["archivo"],
    "group": manifest["tipo"],
    "initial_review": manifest["decision_previa"],
    "requires_bounding_boxes": manifest["requiere_cajas"],
    "source_video": manifest["video_origen"].map(nombre_video),
    "timestamp_seconds": manifest["segundo_aproximado"]
})

manifest_limpio.to_csv(
    DESTINO / "manifest.csv",
    index=False,
    encoding="utf-8-sig"
)

# Documentar el estado inicial
readme = """# Sylva AI — DVR Dataset Stage 2

## Version 0.1: Conjunto de datos sin anotaciones de cajas

Esta versión contiene fotogramas seleccionados a partir de grabaciones de DVR para el proyecto de detección de incendios y humo Sylva AI.

- Candidatos positivos muestreados e imágenes positivas originales:
  incluidos en el manifiesto.
- Candidatos de fondo verificados: incluidos en el manifiesto.
- Anotaciones de cuadros delimitadores (bounding boxes): aún no añadidas.
- Clases previstas para la detección: humo (0), fuego (1).

Las categorías de revisión inicial constituyen metadatos provisionales de preparación de datos; no sustituyen a las etiquetas de cuadros delimitadores (bounding boxes) de YOLO.

Los vídeos de origen no están incluidos en esta instantánea.
"""

(DESTINO / "README.md").write_text(
    readme,
    encoding="utf-8"
)

print("========== DATASET V0.1 PREPARADO ==========")
print("Imágenes copiadas:", len(archivos_copiados))
print("Imágenes en destino:", len(list(CARPETA_IMAGENES.iterdir())))
print("Manifiesto:", DESTINO / "manifest.csv")
print("Destino:", DESTINO)
print(f"Tamaño de imágenes: {sum(p.stat().st_size for p in CARPETA_IMAGENES.iterdir()) / (1024**2):.2f} MiB")

In [ ]:
from pathlib import Path

BASE = Path(
    "/mnt/d/PROYECTOS/INCENDIOS/DATA_PROCESADA_SYLVA/etiquetado_dvr"
)

CARPETA_IMAGENES = BASE / "imagenes_para_anotar"
CARPETA_LABELS = BASE / "labels_yolo"
CARPETA_FONDOS = BASE / "fondos_verificados"

EXTENSIONES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# Buscar imágenes originales
imagenes = [
    p for p in CARPETA_IMAGENES.iterdir()
    if p.is_file() and p.suffix.lower() in EXTENSIONES
]

# Excluir classes.txt del conteo de anotaciones
etiquetas = [
    p for p in CARPETA_LABELS.glob("*.txt")
    if p.name != "classes.txt"
]

# Comparar por nombre de archivo
nombres_etiquetas = {p.stem for p in etiquetas}

sin_etiqueta = [
    img.name for img in imagenes
    if img.stem not in nombres_etiquetas
]

fondos = [
    p for p in CARPETA_FONDOS.iterdir()
    if p.is_file() and p.suffix.lower() in EXTENSIONES
]

print("========== AUDITORÍA DE ETIQUETAS ==========")
print("Imágenes positivas para revisar:", len(imagenes))
print("Archivos de etiquetas YOLO:", len(etiquetas))
print("Imágenes negativas separadas:", len(fondos))
print("Archivo classes.txt existe:", (CARPETA_LABELS / "classes.txt").exists())

print("\nImágenes sin archivo de etiqueta:", len(sin_etiqueta))

if sin_etiqueta:
    for nombre in sin_etiqueta:
        print("FALTA:", nombre)
else:
    print("Todas las imágenes tienen su archivo .txt correspondiente.")

In [ ]:
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd

BASE = Path(
    "/mnt/d/PROYECTOS/INCENDIOS/DATA_PROCESADA_SYLVA/etiquetado_dvr"
)

CARPETA_IMAGENES = BASE / "imagenes_para_anotar"
CARPETA_LABELS = BASE / "labels_yolo"
RUTA_CLASES = CARPETA_LABELS / "classes.txt"
RUTA_MANIFEST = BASE / "manifest_para_etiquetado.csv"

CLASES = {0: "smoke", 1: "fire"}
EXTENSIONES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# 1. Comprobar el orden de las clases
if not RUTA_CLASES.exists():
    raise FileNotFoundError(f"Falta el archivo: {RUTA_CLASES}")

clases_archivo = [
    linea.strip()
    for linea in RUTA_CLASES.read_text(encoding="utf-8").splitlines()
    if linea.strip()
]

print("========== CLASES ==========")
print(clases_archivo)

if clases_archivo != ["smoke", "fire"]:
    print("ADVERTENCIA: el orden de clases no coincide con el esperado.")

# 2. Enumerar imágenes y etiquetas
imagenes = {
    p.stem: p
    for p in CARPETA_IMAGENES.iterdir()
    if p.is_file() and p.suffix.lower() in EXTENSIONES
}

etiquetas = {
    p.stem: p
    for p in CARPETA_LABELS.glob("*.txt")
    if p.name != "classes.txt"
}

faltantes = sorted(set(imagenes) - set(etiquetas))
etiquetas_sin_imagen = sorted(set(etiquetas) - set(imagenes))

objetos_por_clase = Counter()
imagenes_por_clase = Counter()
archivos_vacios = []
errores = []
cajas_fuera_limites = []

# 3. Revisar cada archivo YOLO
for nombre, archivo in etiquetas.items():
    lineas = [
        linea.strip()
        for linea in archivo.read_text(encoding="utf-8").splitlines()
        if linea.strip()
    ]

    if not lineas:
        archivos_vacios.append(nombre)
        continue

    clases_en_imagen = set()

    for numero_linea, linea in enumerate(lineas, start=1):
        partes = linea.split()

        if len(partes) != 5:
            errores.append(
                (archivo.name, numero_linea, "No tiene 5 valores")
            )
            continue

        try:
            clase = int(float(partes[0]))
            xc, yc, w, h = map(float, partes[1:])
        except (ValueError, OverflowError):
            errores.append(
                (archivo.name, numero_linea, "Valores no numéricos")
            )
            continue

        if clase not in CLASES:
            errores.append(
                (archivo.name, numero_linea, f"ID de clase inválido: {clase}")
            )
            continue

        if not np.isfinite([xc, yc, w, h]).all():
            errores.append(
                (archivo.name, numero_linea, "Coordenadas no finitas")
            )
            continue

        if (
            not (0 <= xc <= 1 and 0 <= yc <= 1)
            or not (0 < w <= 1 and 0 < h <= 1)
        ):
            errores.append(
                (archivo.name, numero_linea, "Coordenadas fuera del rango YOLO")
            )
            continue

        if (
            xc - w / 2 < -1e-6
            or yc - h / 2 < -1e-6
            or xc + w / 2 > 1 + 1e-6
            or yc + h / 2 > 1 + 1e-6
        ):
            cajas_fuera_limites.append((archivo.name, numero_linea))

        objetos_por_clase[CLASES[clase]] += 1
        clases_en_imagen.add(CLASES[clase])

    for nombre_clase in clases_en_imagen:
        imagenes_por_clase[nombre_clase] += 1

# 4. Comparar las anotaciones con la revisión visual inicial
faltan_clases_esperadas = []

if RUTA_MANIFEST.exists():
    manifest = pd.read_csv(RUTA_MANIFEST)

    for _, fila in manifest.iterrows():
        tipo = str(fila.get("tipo", ""))
        decision = str(fila.get("decision_previa", "")).strip().lower()
        nombre = Path(str(fila.get("archivo", ""))).stem

        if tipo == "fondo_verificado" or tipo not in {
            "fotograma_positivo",
            "imagen_positiva_original",
        }:
            continue

        if decision == "smoke":
            esperadas = {"smoke"}
        elif decision == "fire":
            esperadas = {"fire"}
        elif decision == "fire_smoke":
            esperadas = {"smoke", "fire"}
        else:
            # Las imágenes originales aún estaban pendientes
            # de confirmar su clase cuando se creó el manifiesto.
            continue

        archivo_label = etiquetas.get(nombre)
        encontradas = set()

        if archivo_label and archivo_label.exists():
            for linea in archivo_label.read_text(encoding="utf-8").splitlines():
                partes = linea.split()
                if len(partes) == 5:
                    try:
                        clase = int(float(partes[0]))
                        if clase in CLASES:
                            encontradas.add(CLASES[clase])
                    except ValueError:
                        pass

        if not esperadas.issubset(encontradas):
            faltan_clases_esperadas.append({
                "archivo": nombre,
                "esperadas": sorted(esperadas),
                "encontradas": sorted(encontradas),
            })

# 5. Mostrar resultados
print("\n========== AUDITORÍA YOLO ==========")
print("Imágenes:", len(imagenes))
print("Etiquetas TXT:", len(etiquetas))
print("Imágenes sin TXT:", len(faltantes))
print("Etiquetas sin imagen:", len(etiquetas_sin_imagen))
print("Archivos TXT vacíos:", len(archivos_vacios))
print("Objetos anotados:", sum(objetos_por_clase.values()))
print("Errores de formato o coordenadas:", len(errores))
print("Cajas fuera de límites:", len(cajas_fuera_limites))

print("\n========== OBJETOS POR CLASE ==========")
print(pd.Series(objetos_por_clase, name="cajas").to_string())

print("\n========== IMÁGENES POR CLASE ==========")
print(pd.Series(imagenes_por_clase, name="imagenes").to_string())

print("\n========== CLASES QUE FALTAN SEGÚN TU REVISIÓN ==========")
print("Registros con discrepancias:", len(faltan_clases_esperadas))
for registro in faltan_clases_esperadas[:20]:
    print(registro)

if faltantes:
    print("\nPrimeras imágenes sin TXT:", faltantes[:10])

if archivos_vacios:
    print("\nPrimeros TXT vacíos:", archivos_vacios[:10])

if errores:
    print("\nPrimeros errores:")
    for error in errores[:10]:
        print(error)

if cajas_fuera_limites:
    print("\nPrimeras cajas fuera de límites:")
    for caja in cajas_fuera_limites[:10]:
        print(caja)

if etiquetas_sin_imagen:
    print("\nEtiquetas sin imagen:", etiquetas_sin_imagen[:10])

Observar conjunto de datos negativos

In [ ]:
from pathlib import Path
import pandas as pd

OUTPUT = Path(
    "/mnt/d/PROYECTOS/INCENDIOS/DATA_PROCESADA_SYLVA"
)

ruta_manifest = OUTPUT / "manifest_frames.csv"
carpeta_negativos = OUTPUT / "candidatos" / "negativos"

manifest = pd.read_csv(ruta_manifest)

# Seleccionar los fotogramas extraídos de vídeos negativos
negativos = manifest[
    manifest["grupo_candidato"] == "negativo"
].copy()

if negativos.empty:
    raise ValueError(
        "No se encontraron fotogramas negativos en el manifiesto."
    )

# Identificar cámara y nombre del vídeo original
negativos["camara"] = negativos["video_origen"].apply(
    lambda x: Path(str(x)).parent.name
)

negativos["video"] = negativos["video_origen"].apply(
    lambda x: Path(str(x)).name
)

# Comprobar cuáles de las imágenes realmente existen
negativos["existe"] = negativos["archivo"].apply(
    lambda x: Path(str(x)).exists()
)

print("========== INVENTARIO DE NEGATIVOS ==========")
print("Fotogramas registrados:", len(negativos))
print("Fotogramas encontrados:", int(negativos["existe"].sum()))
print("Vídeos de origen:", negativos["video_origen"].nunique())
print("Cámaras:", negativos["camara"].nunique())

print("\n========== DISTRIBUCIÓN POR CÁMARA ==========")

resumen = negativos.groupby("camara").agg(
    videos=("video_origen", "nunique"),
    fotogramas=("archivo", "count"),
    archivos_existentes=("existe", "sum")
)

print(resumen.to_string())

print("\n========== VÍDEOS CON MÁS FOTOGRAMAS ==========")

por_video = (
    negativos.groupby(["camara", "video"])
    .size()
    .rename("fotogramas")
    .reset_index()
    .sort_values("fotogramas", ascending=False)
)

print(por_video.head(15).to_string(index=False))

uscar las posibles falsas alarmas automáticamente
Usaremos como detector inicial el YOLOv8s que entrenaste con odiug77/wildfire, que hasta ahora es el mejor modelo con métricas conocidas.

In [ ]:
from pathlib import Path
from ultralytics import YOLO
import pandas as pd
import numpy as np

# =====================================================
# 1. RUTAS
# =====================================================

OUTPUT = Path(
    "/mnt/d/PROYECTOS/INCENDIOS/DATA_PROCESADA_SYLVA"
)

RUTA_MANIFEST = OUTPUT / "manifest_frames.csv"

RUTA_MODELO = (
    Path.home()
    / "IA Vision"
    / "runs"
    / "detect"
    / "yolov8s-odiug77-split"
    / "weights"
    / "best.pt"
)

CARPETA_RESULTADOS = OUTPUT / "analisis_falsas_alarmas"
CARPETA_RESULTADOS.mkdir(parents=True, exist_ok=True)

assert RUTA_MANIFEST.exists(), f"No existe: {RUTA_MANIFEST}"
assert RUTA_MODELO.exists(), f"No existe: {RUTA_MODELO}"

# =====================================================
# 2. CARGAR LOS FOTOGRAMAS NEGATIVOS
# =====================================================

manifest = pd.read_csv(RUTA_MANIFEST)

negativos = manifest[
    manifest["grupo_candidato"] == "negativo"
].copy()

negativos["archivo"] = negativos["archivo"].astype(str)

negativos = negativos[
    negativos["archivo"].map(
        lambda x: Path(x).is_file()
    )
].reset_index(drop=True)

if negativos.empty:
    raise ValueError("No hay imágenes negativas disponibles.")

print("Fotogramas disponibles:", len(negativos))
print("Vídeos de origen:", negativos["video_origen"].nunique())

# =====================================================
# 3. CARGAR EL MODELO
# =====================================================

modelo = YOLO(str(RUTA_MODELO))

print("Modelo:", RUTA_MODELO)
print("Clases:", modelo.names)

# Identificar los IDs a partir de los nombres de las clases
names = modelo.names

if isinstance(names, dict):
    nombre_por_id = {
        int(k): str(v).lower()
        for k, v in names.items()
    }
else:
    nombre_por_id = {
        i: str(v).lower()
        for i, v in enumerate(names)
    }

id_smoke = next(
    (i for i, nombre in nombre_por_id.items()
     if nombre == "smoke"),
    None
)

id_fire = next(
    (i for i, nombre in nombre_por_id.items()
     if nombre == "fire"),
    None
)

if id_smoke is None or id_fire is None:
    raise ValueError(
        f"El modelo no tiene las clases smoke y fire: {nombre_por_id}"
    )

# =====================================================
# 4. INFERENCIA SOBRE LAS IMÁGENES NEGATIVAS
# =====================================================

CONF_MIN = 0.05
TAMANO_LOTE = 32

# Asociar ruta de cada imagen con sus datos de origen
metadata = {}

for _, fila in negativos.iterrows():
    ruta = str(Path(fila["archivo"]).resolve())
    metadata[ruta] = fila.to_dict()

resultados = []

rutas = negativos["archivo"].tolist()

for inicio in range(0, len(rutas), TAMANO_LOTE):

    lote = rutas[inicio:inicio + TAMANO_LOTE]

    predicciones = modelo.predict(
        source=lote,
        imgsz=640,
        batch=8,
        device=0,
        conf=CONF_MIN,
        iou=0.7,
        max_det=100,
        verbose=False,
        stream=True,
        save=False
    )

    for pred in predicciones:

        ruta = str(Path(pred.path).resolve())
        info = metadata.get(ruta, {})

        conf_smoke = []
        conf_fire = []

        if pred.boxes is not None and len(pred.boxes) > 0:

            clases_pred = (
                pred.boxes.cls.detach().cpu().numpy().astype(int)
            )

            conf_pred = (
                pred.boxes.conf.detach().cpu().numpy()
            )

            for clase, confianza in zip(clases_pred, conf_pred):

                if clase == id_smoke:
                    conf_smoke.append(float(confianza))

                elif clase == id_fire:
                    conf_fire.append(float(confianza))

        max_smoke = max(conf_smoke, default=0.0)
        max_fire = max(conf_fire, default=0.0)

        resultados.append({
            "archivo": ruta,
            "camara": Path(
                str(info.get("video_origen", "desconocida"))
            ).parent.name,
            "video_origen": info.get("video_origen", ""),
            "segundo_aproximado": info.get(
                "segundo_aproximado", np.nan
            ),
            "detecciones_smoke": len(conf_smoke),
            "detecciones_fire": len(conf_fire),
            "max_conf_smoke": max_smoke,
            "max_conf_fire": max_fire,
            "max_conf_general": max(max_smoke, max_fire)
        })

    procesadas = min(inicio + len(lote), len(rutas))

    print(
        f"Procesadas {procesadas}/{len(rutas)} imágenes",
        end="\r"
    )

print("\nInferencia terminada.")

# =====================================================
# 5. GUARDAR LOS RESULTADOS
# =====================================================

df_resultados = pd.DataFrame(resultados)

ruta_todos = CARPETA_RESULTADOS / "negativos_evaluados.csv"

df_resultados.to_csv(
    ruta_todos,
    index=False,
    encoding="utf-8-sig"
)

# Candidatos con predicciones
candidatos = df_resultados[
    (df_resultados["detecciones_smoke"] > 0)
    | (df_resultados["detecciones_fire"] > 0)
].copy()

candidatos = candidatos.sort_values(
    "max_conf_general",
    ascending=False
)

ruta_candidatos = (
    CARPETA_RESULTADOS / "posibles_falsas_alarmas.csv"
)

candidatos.to_csv(
    ruta_candidatos,
    index=False,
    encoding="utf-8-sig"
)

# =====================================================
# 6. RESUMEN
# =====================================================

print("\n========== RESULTADO DE HARD-NEGATIVE MINING ==========")

print("Imágenes analizadas:", len(df_resultados))
print("Imágenes con detecciones:", len(candidatos))
print(
    "Imágenes con detecciones de humo:",
    int((df_resultados["detecciones_smoke"] > 0).sum())
)
print(
    "Imágenes con detecciones de fuego:",
    int((df_resultados["detecciones_fire"] > 0).sum())
)

print("\n========== POSIBLES FALSAS ALARMAS POR CÁMARA ==========")

resumen = df_resultados.groupby("camara").agg(
    fotogramas=("archivo", "count"),
    con_detecciones=(
        "max_conf_general",
        lambda s: int((s > 0).sum())
    ),
    confianza_maxima=("max_conf_general", "max")
)

print(resumen.round(3).to_string())

print("\n========== TOP 30 CANDIDATOS ==========")

columnas = [
    "camara",
    "video_origen",
    "segundo_aproximado",
    "max_conf_smoke",
    "max_conf_fire",
    "archivo"
]

print(
    candidatos[columnas]
    .head(30)
    .round(3)
    .to_string(index=False)
)

print("\nCSV de todas las evaluaciones:", ruta_todos)
print("CSV de candidatos:", ruta_candidatos)

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import math

from ultralytics import YOLO
from IPython.display import display

# =====================================================
# 1. RUTAS
# =====================================================

OUTPUT = Path(
    "/mnt/d/PROYECTOS/INCENDIOS/DATA_PROCESADA_SYLVA"
)

CARPETA_RESULTADOS = OUTPUT / "analisis_falsas_alarmas"

CSV_CANDIDATOS = (
    CARPETA_RESULTADOS / "posibles_falsas_alarmas.csv"
)

RUTA_MODELO = (
    Path.home()
    / "IA Vision"
    / "runs"
    / "detect"
    / "yolov8s-odiug77-split"
    / "weights"
    / "best.pt"
)

assert CSV_CANDIDATOS.exists(), f"No existe: {CSV_CANDIDATOS}"
assert RUTA_MODELO.exists(), f"No existe: {RUTA_MODELO}"

df_candidatos = pd.read_csv(CSV_CANDIDATOS)

# Solo considerar candidatos cuyos archivos existan
df_candidatos = df_candidatos[
    df_candidatos["archivo"].map(
        lambda x: Path(str(x)).is_file()
    )
].copy()

# =====================================================
# 2. SELECCIONAR LOS 6 MÁS CONFIADOS POR CÁMARA
# =====================================================

seleccion = (
    df_candidatos
    .sort_values("max_conf_general", ascending=False)
    .groupby("camara", group_keys=False)
    .head(6)
    .sort_values(
        ["camara", "max_conf_general"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

seleccion["revision_manual"] = ""
seleccion["observaciones"] = ""

print("========== MUESTRA PARA REVISIÓN ==========")
print("Candidatos seleccionados:", len(seleccion))
print(
    seleccion.groupby("camara")
    .size()
    .rename("imágenes")
    .to_string()
)

print("\nArchivo | Cámara | Segundo | Confianza humo | Confianza fuego")

print(
    seleccion[
        [
            "archivo",
            "camara",
            "segundo_aproximado",
            "max_conf_smoke",
            "max_conf_fire",
        ]
    ].round(3).to_string(index=False)
)

# Guardar registro de revisión para uso posterior
CSV_REVISION = (
    CARPETA_RESULTADOS
    / "revision_manual_top_candidatos.csv"
)

seleccion.to_csv(
    CSV_REVISION,
    index=False,
    encoding="utf-8-sig"
)

# =====================================================
# 3. VOLVER A EJECUTAR EL MODELO SOBRE ESTA MUESTRA
#    PARA MOSTRAR LAS CAJAS PREDICHAS
# =====================================================

modelo_revision = YOLO(str(RUTA_MODELO))

rutas = seleccion["archivo"].astype(str).tolist()

predicciones = modelo_revision.predict(
    source=rutas,
    imgsz=640,
    batch=8,
    device=0,
    conf=0.05,
    iou=0.7,
    max_det=100,
    verbose=False,
    save=False
)

# =====================================================
# 4. MOSTRAR MOSAICOS POR CÁMARA
# =====================================================

registros = seleccion.to_dict("records")

for camara in seleccion["camara"].drop_duplicates():

    indices = [
        i for i, registro in enumerate(registros)
        if registro["camara"] == camara
    ]

    if not indices:
        continue

    columnas = 3
    filas = math.ceil(len(indices) / columnas)

    fig, axes = plt.subplots(
        filas,
        columnas,
        figsize=(15, filas * 4.5),
        squeeze=False
    )

    for ax in axes.flat:
        ax.axis("off")

    for ax, i in zip(axes.flat, indices):

        registro = registros[i]
        resultado = predicciones[i]

        # Ultralytics entrega la imagen dibujada en BGR
        imagen = resultado.plot()
        imagen = cv2.cvtColor(
            imagen,
            cv2.COLOR_BGR2RGB
        )

        ax.imshow(imagen)

        nombre_video = Path(
            str(registro["video_origen"])
        ).name

        segundo = registro["segundo_aproximado"]

        ax.set_title(
            f"{nombre_video}\nt={segundo}s | "
            f"smoke={registro['max_conf_smoke']:.2f}, "
            f"fire={registro['max_conf_fire']:.2f}",
            fontsize=8
        )

        ax.axis("off")

    fig.suptitle(
        f"{camara}: candidatos con mayor confianza",
        fontsize=14
    )

    plt.tight_layout()
    display(fig)
    plt.close(fig)

print("\nRegistro de revisión guardado en:")
print(CSV_REVISION)

In [ ]:
from pathlib import Path
import pandas as pd
import shutil

# =====================================================
# 1. RUTAS
# =====================================================

OUTPUT = Path(
    "/mnt/d/PROYECTOS/INCENDIOS/DATA_PROCESADA_SYLVA"
)

CSV_CANDIDATOS = (
    OUTPUT
    / "analisis_falsas_alarmas"
    / "revision_manual_top_candidatos.csv"
)

DESTINO = OUTPUT / "etiquetado_dvr" / "hard_negatives_cloud"

CARPETA_IMAGENES = DESTINO / "images"
CARPETA_LABELS = DESTINO / "labels"

assert CSV_CANDIDATOS.exists(), (
    f"No se encontró el CSV: {CSV_CANDIDATOS}"
)

CARPETA_IMAGENES.mkdir(parents=True, exist_ok=True)
CARPETA_LABELS.mkdir(parents=True, exist_ok=True)

# =====================================================
# 2. LEER LOS CANDIDATOS REVISADOS
# =====================================================

df = pd.read_csv(CSV_CANDIDATOS)

if "archivo" not in df.columns:
    raise ValueError("El CSV no contiene la columna 'archivo'.")

print("Candidatos registrados:", len(df))

# No asumir que todas las imágenes del CSV se revisaron.
# Confirmar que la selección corresponde a la revisión visual.
if "revision_manual" not in df.columns:
    df["revision_manual"] = ""

if "observaciones" not in df.columns:
    df["observaciones"] = ""

print(
    "\nCompleta la confirmación en el CSV antes de continuar "
    "si todavía no has revisado todas las imágenes."
)

# En este caso confirmamos las imágenes revisadas visualmente.
# La confirmación se limita a la selección de este archivo.
df["revision_manual"] = "background_cloud_confirmed"
df["observaciones"] = (
    "Nube sin humo ni fuego visible; falso positivo del modelo "
    "confirmado mediante revisión visual."
)

# =====================================================
# 3. COPIAR IMÁGENES Y CREAR ETIQUETAS VACÍAS
# =====================================================

copiadas = 0
errores = []

for _, fila in df.iterrows():

    origen = Path(str(fila["archivo"]))

    if not origen.exists():
        errores.append(str(origen))
        continue

    # El nombre original incluye cámara, vídeo y timestamp.
    nombre = origen.name

    destino_imagen = CARPETA_IMAGENES / nombre
    destino_label = CARPETA_LABELS / f"{origen.stem}.txt"

    # No sobrescribir imágenes o etiquetas existentes.
    if destino_imagen.exists() or destino_label.exists():
        print("Ya existe, se omite:", nombre)
        continue

    shutil.copy2(origen, destino_imagen)

    # Archivo vacío = imagen de fondo sin objetos anotados.
    destino_label.write_text("", encoding="utf-8")

    copiadas += 1

# =====================================================
# 4. GUARDAR REGISTRO DE LA REVISIÓN
# =====================================================

ruta_revision_final = DESTINO / "hard_negatives_manifest.csv"

df["tipo_confirmado"] = "negativo_dificil_nubes"
df["requiere_cajas"] = False

df.to_csv(
    ruta_revision_final,
    index=False,
    encoding="utf-8-sig"
)

# =====================================================
# 5. RESUMEN
# =====================================================

imagenes_guardadas = list(CARPETA_IMAGENES.iterdir())
labels_guardadas = list(CARPETA_LABELS.glob("*.txt"))

print("\n========== NEGATIVOS DIFÍCILES ==========")
print("Imágenes copiadas en esta ejecución:", copiadas)
print("Imágenes en la carpeta:", len(imagenes_guardadas))
print("Etiquetas YOLO vacías:", len(labels_guardadas))
print("Archivos fuente no encontrados:", len(errores))

print("\nCarpeta de imágenes:")
print(CARPETA_IMAGENES)

print("\nCarpeta de etiquetas:")
print(CARPETA_LABELS)

print("\nManifiesto:")
print(ruta_revision_final)

if errores:
    print("\nArchivos que no se encontraron:")
    for error in errores:
        print(error)

Integrar el nuevo dataser al repositorio GIT

In [ ]:
from pathlib import Path
import pandas as pd
import shutil
import numpy as np

# ============================================================
# 1. RUTAS
# ============================================================

OUTPUT = Path(
    "/mnt/d/PROYECTOS/INCENDIOS/DATA_PROCESADA_SYLVA"
)

BASE_ETIQUETADO = OUTPUT / "etiquetado_dvr"

# Etiquetas de las imágenes positivas
SRC_LABELS = BASE_ETIQUETADO / "labels_yolo"

# Imágenes negativas verificadas
SRC_FONDOS = BASE_ETIQUETADO / "fondos_verificados"

# Negativos difíciles: nubes que activaron el detector
SRC_HARD_IMAGES = (
    BASE_ETIQUETADO / "hard_negatives_cloud" / "images"
)

SRC_HARD_LABELS = (
    BASE_ETIQUETADO / "hard_negatives_cloud" / "labels"
)

# Manifiestos de origen
SRC_MANIFEST = (
    BASE_ETIQUETADO / "manifest_para_etiquetado.csv"
)

SRC_HARD_MANIFEST = (
    BASE_ETIQUETADO
    / "hard_negatives_cloud"
    / "hard_negatives_manifest.csv"
)

# Repositorio Git local
REPO = Path.home() / "sylva-ai-fire-smoke"
DESTINO = REPO / "dataset_versions" / "dvr_stage2"

DST_IMAGES = DESTINO / "images"
DST_LABELS = DESTINO / "labels"

for ruta in [
    REPO,
    SRC_LABELS,
    SRC_FONDOS,
    SRC_HARD_IMAGES,
    SRC_HARD_LABELS,
    SRC_MANIFEST,
    SRC_HARD_MANIFEST,
    DST_IMAGES,
]:
    if not ruta.exists():
        raise FileNotFoundError(f"No se encontró: {ruta}")

DST_LABELS.mkdir(parents=True, exist_ok=True)

EXTENSIONES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


# ============================================================
# 2. LEER MANIFIESTOS
# ============================================================

manifest = pd.read_csv(SRC_MANIFEST)
hard_manifest = pd.read_csv(SRC_HARD_MANIFEST)

print("Filas del manifiesto original:", len(manifest))
print("Filas del manifiesto de nubes:", len(hard_manifest))


# ============================================================
# 3. COPIAR LOS 30 NEGATIVOS DIFÍCILES
# ============================================================

hard_images = sorted(
    p for p in SRC_HARD_IMAGES.iterdir()
    if p.is_file() and p.suffix.lower() in EXTENSIONES
)

hard_labels = {
    p.stem: p
    for p in SRC_HARD_LABELS.glob("*.txt")
}

copiados_hard = 0

for imagen in hard_images:
    destino_img = DST_IMAGES / imagen.name
    destino_lbl = DST_LABELS / f"{imagen.stem}.txt"

    if not destino_img.exists():
        shutil.copy2(imagen, destino_img)
        copiados_hard += 1
    elif imagen.resolve() != destino_img.resolve():
        # Evitar sobrescribir archivos existentes sin verificar
        if imagen.stat().st_size != destino_img.stat().st_size:
            raise FileExistsError(
                f"Hay un archivo con el mismo nombre y distinto tamaño: "
                f"{destino_img}"
            )

    etiqueta_src = hard_labels.get(imagen.stem)

    if etiqueta_src is None:
        raise FileNotFoundError(
            f"Falta la etiqueta negativa de {imagen.name}"
        )

    # Estos negativos deben tener etiquetas vacías
    contenido = etiqueta_src.read_text(encoding="utf-8").strip()

    if contenido:
        raise ValueError(
            f"El negativo tiene cajas y necesita revisión: {imagen.name}"
        )

    if not destino_lbl.exists():
        shutil.copy2(etiqueta_src, destino_lbl)
    elif destino_lbl.read_text(encoding="utf-8").strip():
        raise ValueError(
            f"La etiqueta de destino no está vacía: {destino_lbl}"
        )


# ============================================================
# 4. COPIAR LAS 109 ETIQUETAS POSITIVAS
# ============================================================

labels_positivas = [
    p for p in SRC_LABELS.glob("*.txt")
    if p.name != "classes.txt"
]

copiadas_labels = 0

for etiqueta in labels_positivas:
    imagen_destino = next(
        (
            DST_IMAGES / f"{etiqueta.stem}{extension}"
            for extension in EXTENSIONES
            if (DST_IMAGES / f"{etiqueta.stem}{extension}").exists()
        ),
        None
    )

    if imagen_destino is None:
        raise FileNotFoundError(
            f"La imagen etiquetada no está en el repositorio: "
            f"{etiqueta.stem}"
        )

    destino_lbl = DST_LABELS / etiqueta.name

    if destino_lbl.exists():
        # Evitar reemplazar anotaciones diferentes sin revisión
        if destino_lbl.read_text(encoding="utf-8") != \
                etiqueta.read_text(encoding="utf-8"):
            raise FileExistsError(
                f"Ya existe una etiqueta diferente: {destino_lbl}"
            )
    else:
        shutil.copy2(etiqueta, destino_lbl)
        copiadas_labels += 1


# ============================================================
# 5. CREAR ETIQUETAS VACÍAS PARA LOS 25 FONDOS
# ============================================================

fondos = sorted(
    p for p in SRC_FONDOS.iterdir()
    if p.is_file() and p.suffix.lower() in EXTENSIONES
)

creadas_fondos = 0

for imagen in fondos:
    destino_img = DST_IMAGES / imagen.name
    destino_lbl = DST_LABELS / f"{imagen.stem}.txt"

    if not destino_img.exists():
        # Algunos fondos podrían no estar en la copia inicial
        shutil.copy2(imagen, destino_img)

    if destino_lbl.exists():
        if destino_lbl.read_text(encoding="utf-8").strip():
            raise ValueError(
                f"El fondo tiene cajas inesperadas: {destino_lbl}"
            )
    else:
        destino_lbl.write_text("", encoding="utf-8")
        creadas_fondos += 1


# ============================================================
# 6. ACTUALIZAR EL MANIFIESTO SIN DUPLICAR ARCHIVOS
# ============================================================

hard_rows = []

for _, fila in hard_manifest.iterrows():
    ruta_src = Path(str(fila["archivo"]))
    filename = ruta_src.name

    hard_rows.append({
        "filename": filename,
        "group": "hard_negative_cloud",
        "initial_review": "background_cloud_confirmed",
        "requires_bounding_boxes": False,
        "source_video": Path(
            str(fila["video_origen"])
        ).name,
        "timestamp_seconds": fila.get(
            "segundo_aproximado", np.nan
        ),
    })

hard_rows_df = pd.DataFrame(hard_rows)

manifest_actualizado = pd.concat(
    [manifest, hard_rows_df],
    ignore_index=True
)

manifest_actualizado = manifest_actualizado.drop_duplicates(
    subset=["filename"],
    keep="last"
)

ruta_manifest_destino = DESTINO / "manifest.csv"

manifest_actualizado.to_csv(
    ruta_manifest_destino,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 7. AUDITORÍA FINAL DE IMÁGENES Y ETIQUETAS
# ============================================================

imagenes_destino = {
    p.stem: p
    for p in DST_IMAGES.iterdir()
    if p.is_file() and p.suffix.lower() in EXTENSIONES
}

etiquetas_destino = {
    p.stem: p
    for p in DST_LABELS.glob("*.txt")
}

sin_etiqueta = sorted(
    set(imagenes_destino) - set(etiquetas_destino)
)

labels_sin_imagen = sorted(
    set(etiquetas_destino) - set(imagenes_destino)
)

positivas_vacias = [
    e.name
    for e in labels_positivas
    if (
        (DST_LABELS / e.name).exists()
        and not (DST_LABELS / e.name).read_text(
            encoding="utf-8"
        ).strip()
    )
]

print("\n========== DATASET V0.2 ==========")
print("Imágenes positivas etiquetadas:", len(labels_positivas))
print("Fondos negativos:", len(fondos))
print("Nubes difíciles:", len(hard_images))
print("Imágenes en el repositorio:", len(imagenes_destino))
print("Archivos TXT:", len(etiquetas_destino))
print("Etiquetas sin imagen:", len(labels_sin_imagen))
print("Imágenes sin etiqueta:", len(sin_etiqueta))
print("Positivos con etiqueta vacía:", len(positivas_vacias))

print("\nManifiesto actualizado:", ruta_manifest_destino)

if sin_etiqueta or labels_sin_imagen or positivas_vacias:
    print(
        "\nATENCIÓN: revisa las discrepancias antes "
        "de registrar la nueva versión."
    )
else:
    print(
        "\nAuditoría estructural correcta. "
        "El dataset está preparado para revisión final."
    )